<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/Python-2026/Practice/Scikit_learn_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Вариант 1. Scikit-learn — полный курс (110 задач: setup + 105 основных)

> **Тема:** библиотека — предсказание возврата книги вовремя, прогноз штрафа, кластеризация читателей, feature importance.
> **Цель:** покрыть **всю базовую и продвинутую теорию** ML на табличных данных: препроцессинг → baseline → метрики → модели → кросс-валидация → тюнинг → unsupervised → feature engineering → imbalanced → calibration → production → интерпретация → финальный проект.
> **Стек:** `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`, `statsmodels` (для сравнения), `shap` (обзорно), `joblib`, `imbalanced-learn` (обзорно).
> **Данные (сквозные):** `loans.csv` — таблица выдач. Две задачи: **классификация** (`returned_on_time: 0/1`) и **регрессия** (`fine`). Плюс `make_classification`/`make_regression` для контролируемых экспериментов.

---

## 🟢 Уровень 0. Setup: синтетические данные (0.1–0.5)

**Что изучается:** проектирование ML-задачи, генерация через `sklearn.datasets`, контролируемые сигналы, дисбаланс классов, пропуски, `train_test_split`, проверка «данные не слишком лёгкие».

**Задача 0.1 (спроектировать ML-задачу).** Прежде чем писать `model.fit()`, нужно понять **что** предсказываем и **зачем**. Сформулируйте две задачи: (а) **классификация** — предсказать `returned_on_time` (0/1) по признакам `days_held`, `genre`, `hall`, `reader_age`, `book_price`; (б) **регрессия** — предсказать `fine` по тем же признакам. Для каждой: целевая переменная, признаки, метрика (accuracy/F1 для классификации, RMSE/MAE для регрессии), baseline. Ожидаемый результат: текстовое описание + таблица «признак → тип → связь с target». В комментариях поясните: чем классификация отличается от регрессии; почему метрика выбирается **до** модели (F1 при дисбалансе, RMSE при симметричных ошибках); почему «угадать среднее» — обязательный baseline; что такое data leakage (признак, который «знает» ответ). Подсказка: нарисуйте DAG: `genre → days_held → returned_on_time`; `days_overdue → fine`.

**Задача 0.2 (сгенерировать `loans.csv`).** Сгенерируйте `n = 2000` строк: `days_held ~ lognormal(μ=2.5, σ=0.4)`, `days_overdue ~ zero-inflated exponential`, `genre` (4 категории), `hall` (3 категории), `reader_age ~ normal(35, 12)`, `book_price ~ uniform(200, 2000)`. Целевые: `returned_on_time ~ Binomial(1, p)` где `p = sigmoid(2 − 0.1·days_held + 0.3·reader_age/10 + noise)`; `fine = days_overdue * 50 + noise`. `seed = 42` через `np.random.default_rng`. Ожидаемый вывод: `df.shape == (2000, 8)`; `returned_on_time.mean() ≈ 0.6`; корреляция `days_held` и `returned_on_time` отрицательная (`≈ −0.4`). В комментариях поясните: почему `lognormal` для `days_held` (правый хвост, реальные сроки); почему `sigmoid` для вероятности (гарантирует `[0,1]`); зачем `seed` (воспроизводимость); почему дисбаланс 60/40 — ещё не проблема, но 95/5 — уже да. Подсказка: `p = 1 / (1 + np.exp(-(2 - 0.1*days_held + 0.3*reader_age/10 + rng.normal(0, 0.5, n))))`.

**Задача 0.3 (сгенерировать через `make_classification` и `make_regression`).** Для контролируемых экспериментов используйте: (а) `make_classification(n_samples=1000, n_features=10, n_informative=5, n_redundant=2, n_classes=2, flip_y=0.05, random_state=42)`; (б) `make_regression(n_samples=1000, n_features=10, n_informative=5, noise=10, random_state=42)`. Ожидаемый вывод: `X.shape == (1000, 10)`; `y.mean() ≈ 0.5`; `flip_y = 5%` — шум в метках; регрессия — `R² ≈ 0.9` у идеальной модели. В комментариях поясните: что значит `n_informative` (реально полезные признаки), `n_redundant` (линейные комбинации информативных — модель не должна их «разделять»), `flip_y` (шум в метках — accuracy >95% невозможна); зачем такие данные (проверка, что модель работает, когда «истина» известна); как связаны `n_informative` и оптимальная accuracy (~95% потолок при 5 информативных). Подсказка: `from sklearn.datasets import make_classification, make_regression`.

**Задача 0.4 (добавить «грязь»: пропуски, выбросы, дисбаланс).** В `loans.csv` добавьте: 3% пропусков в `reader_age`, 1% в `book_price`; 2% выбросов в `fine` (`= 99999`); дисбаланс — уберите 30% положительных классов (`returned_on_time = 1`). Ожидаемый вывод: `df.isna().sum().sum() ≈ 80`; `df["returned_on_time"].mean() ≈ 0.42`; `df["fine"].max() == 99999`. В комментариях поясните: почему пропуски ломают `model.fit()` (`ValueError: Input contains NaN`); почему выбросы ломают `StandardScaler` (среднее и std «раздуваются»); как дисбаланс влияет на accuracy (модель «всегда 0» даёт 58%); что делать с дисбалансом (`class_weight="balanced"`, `SMOTE`, threshold tuning — задача 92). Подсказка: `df.loc[rng.choice(df.index, 60, replace=False), "reader_age"] = np.nan`.

**Задача 0.5 (`train_test_split` + проверка «данные не слишком лёгкие»).** Разделите `loans.csv` на train/test: `test_size=0.2`, `stratify=y`, `random_state=42`. Обучите `DummyClassifier(strategy="most_frequent")` — baseline. Ожидаемый вывод: `train: 1600, test: 400`; `Dummy accuracy = 0.58`; проверка: `y_train.mean() ≈ y_test.mean()`. В комментариях поясните: зачем `stratify` (сохранить пропорцию классов); почему `random_state` обязателен; что такое baseline и почему без него нельзя оценить модель; почему «данные не слишком лёгкие» — если `Dummy` даёт 0.95, задача тривиальна; как проверить, что split не «сдвинул» распределение. Подсказка: `from sklearn.model_selection import train_test_split; X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)`.

---

## 🟢 Уровень 1. Препроцессинг (1–10)

**Что изучается:** `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `OneHotEncoder`, `OrdinalEncoder`, `SimpleImputer`, `KNNImputer`, `PolynomialFeatures`, `ColumnTransformer`, `Pipeline` (базово), data leakage при препроцессинге.

**Задача 1 (`StandardScaler`: база).** Признаки `days_held` и `book_price` в разных шкалах — линейные модели это «пугает». Примените `StandardScaler` к числовым признакам. Ожидаемый вывод: `mean ≈ 0`, `std ≈ 1` после `fit_transform`; `scaler.mean_` и `scaler.scale_` показывают параметры. В комментариях поясните: что делает `z = (x − μ) / σ`; почему `fit` только на train (иначе data leakage); почему `fit_transform` на train, `transform` на test; почему `StandardScaler` не меняет форму распределения (только сдвиг и масштаб). Подсказка: `from sklearn.preprocessing import StandardScaler; scaler = StandardScaler(); X_train_scaled = scaler.fit_transform(X_train[["days_held", "book_price"]]); X_test_scaled = scaler.transform(X_test[["days_held", "book_price"]])`.

**Задача 2 (`MinMaxScaler` + разница).** Примените `MinMaxScaler` к тем же признакам. Сравните с `StandardScaler`: что изменилось? Ожидаемый вывод: `min = 0`, `max = 1`; распределение сжато в `[0, 1]`, но форма сохранена. В комментариях поясните: когда `MinMaxScaler` лучше `StandardScaler` (нейросети, признаки с границами); почему `MinMaxScaler` чувствителен к выбросам (один выброс сжимает остальные в «точку»); почему для линейных моделей — `StandardScaler`, для деревьев — не важно. Подсказка: `from sklearn.preprocessing import MinMaxScaler`.

**Задача 3 (`RobustScaler`).** В `days_held` есть выбросы — `StandardScaler` «раздувается». Примените `RobustScaler` (медиана + IQR). Ожидаемый вывод: `RobustScaler` не меняет масштаб остальных точек; `StandardScaler` — «сплющивает». В комментариях поясните: чем `RobustScaler` отличается (`(x − median) / IQR`); когда он незаменим (выбросы, тяжёлые хвосты); почему деревьям он не нужен, но и не вредит; как сравнить распределения до/после (`sns.histplot`). Подсказка: `from sklearn.preprocessing import RobustScaler`.

**Задача 4 (`OneHotEncoder`).** Признаки `genre` и `hall` — категориальные. Закодируйте через `OneHotEncoder`. Ожидаемый вывод: `shape (n, 7)` — 4 жанра + 3 зала (без baseline-категории при `drop="first"`). В комментариях поясните: что делает one-hot (каждая категория — отдельный бинарный столбец); зачем `drop="first"` (избежать мультиколлинеарности, dummy variable trap); чем `OneHotEncoder` лучше `pd.get_dummies` (сохраняет соответствие train/test, не теряет категории при `transform`); почему high-cardinality признаки (1000+ категорий) плохо кодировать one-hot (проклятие размерности). Подсказка: `from sklearn.preprocessing import OneHotEncoder; enc = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")`.

**Задача 5 (`OrdinalEncoder`).** Для порядковых категорий (`small/medium/large` или `low/mid/high`) one-hot теряет порядок. Примените `OrdinalEncoder`. Ожидаемый вывод: `small → 0, medium → 1, large → 2`. В комментариях поясните: когда `OrdinalEncoder` лучше `OneHotEncoder` (есть **истинный** порядок); почему для номинальных категорий (жанр, зал) `OrdinalEncoder` — ошибка (модель подумает, что «детектив > роман»); как задать порядок вручную (`categories=[["small", "medium", "large"]]`); где это применимо (уровень образования, тариф). Подсказка: `from sklearn.preprocessing import OrdinalEncoder; enc = OrdinalEncoder(categories=[["low", "mid", "high"]])`.

**Задача 6 (`SimpleImputer`).** В `reader_age` 3% пропусков. Заполните через `SimpleImputer` тремя стратегиями: `mean`, `median`, `most_frequent`. Сравните. Ожидаемый вывод: `mean` вставляет среднее train, `median` — медиану (устойчива к выбросам), `most_frequent` — моду. В комментариях поясните: почему `mean` плох при выбросах; почему нельзя вставлять среднее по всему датасету (data leakage — `fit` только на train); когда лучше `KNNImputer` (зависимости между признаками) или `IterativeImputer` (MICE); почему иногда «пропуск — это сигнал» (добавить `is_missing` как отдельный признак). Подсказка: `from sklearn.impute import SimpleImputer; imp = SimpleImputer(strategy="median"); X_imputed = imp.fit_transform(X)`.

**Задача 7 (`KNNImputer`).** Пропуски в `reader_age` и `book_price` коррелируют. Примените `KNNImputer(n_neighbors=5)`. Ожидаемый вывод: заполненные значения ближе к реальным, чем у `SimpleImputer`. В комментариях поясните: как работает `KNNImputer` (находит `k` соседей по остальным признакам, берёт среднее); почему он лучше `SimpleImputer` при зависимостях; почему требует стандартизации; когда он **хуже** (`n` маленькое, много пропусков, категориальные признаки); как выбрать `k`. Подсказка: `from sklearn.impute import KNNImputer`.

**Задача 8 (`PolynomialFeatures`).** Линейная модель не поймает нелинейность. Добавьте полиномиальные признаки (`degree=2`) к `days_held` и `reader_age`. Ожидаемый вывод: `3 признака → 6` (`x1, x2, x1², x1·x2, x2², 1`). В комментариях поясните: что делает `PolynomialFeatures`; зачем (линейная модель + полином = нелинейная); почему `degree=2` уже даёт много признаков (`d=10 → 65`); как бороться с размерностью (`Ridge`, `Lasso`, `SelectKBest`); почему для деревьев полиномы не нужны (они и так находят нелинейность). Подсказка: `from sklearn.preprocessing import PolynomialFeatures; poly = PolynomialFeatures(degree=2, include_bias=False)`.

**Задача 9 (`ColumnTransformer`).** Признаки разные: числовые (`days_held`, `book_price`), категориальные (`genre`, `hall`). К ним нужны **разные** препроцессоры. Примените `ColumnTransformer`. Ожидаемый вывод: `shape (n, 2 + 4 + 3)` — 2 числовых + 4 one-hot жанра + 3 one-hot зала. В комментариях поясните: что делает `ColumnTransformer` (применяет разные трансформеры к разным столбцам); зачем это нужно (нельзя один `StandardScaler` применить к категориям); как `remainder="passthrough"` оставляет остальные столбцы; почему это **обязательный** шаг для production (`Pipeline` + `ColumnTransformer` — стандарт). Подсказка: `from sklearn.compose import ColumnTransformer; ct = ColumnTransformer([("num", StandardScaler(), ["days_held", "book_price"]), ("cat", OneHotEncoder(drop="first"), ["genre", "hall"])])`.

**Задача 10 (`Pipeline`: база).** Соберите препроцессинг в `Pipeline`: imputer → scaler → one-hot → logistic regression. Ожидаемый вывод: `pipeline.fit(X_train, y_train)`, `pipeline.predict(X_test)` — всё одной строкой. В комментариях поясните: что такое `Pipeline` (цепочка «трансформер → трансформер → модель»); зачем (нет data leakage — `fit` только на train, применяется ко всему автоматически); почему `pipeline.named_steps` полезен (доступ к шагам); как `Pipeline` + `GridSearchCV` даёт честный тюнинг (см. задачу 66); почему **всегда** используйте `Pipeline` в production. Подсказка: `from sklearn.pipeline import Pipeline; pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", LogisticRegression())])`.

---

## 🔵 Уровень 2. Splits, CV, baseline (11–18)

**Что изучается:** `train_test_split`, `DummyClassifier`/`DummyRegressor`, `KFold`, `StratifiedKFold`, `GroupKFold`, `TimeSeriesSplit`, `cross_val_score`, `cross_validate`.

**Задача 11 (`DummyClassifier`: baseline).** Прежде чем обучать модель, нужен «тупой» baseline. Сравните три стратегии: `most_frequent`, `stratified`, `uniform`. Ожидаемый вывод: `most_frequent: accuracy = 0.58`; `stratified: accuracy ≈ 0.50`; `uniform: accuracy ≈ 0.50`. В комментариях поясните: зачем baseline (модель должна **побить** baseline); почему `most_frequent` на дисбалансе даёт «обманчиво высокий» accuracy; почему `stratified` — честный baseline для дисбаланса; когда baseline уже «достаточно хорош» (если модель не бьёт его — что-то не так с данными). Подсказка: `from sklearn.dummy import DummyClassifier; DummyClassifier(strategy="stratified").fit(X_train, y_train).score(X_test, y_test)`.

**Задача 12 (`DummyRegressor`).** Для регрессии — `DummyRegressor(strategy="mean")` (всегда предсказывает среднее). Ожидаемый вывод: `RMSE(Dummy) = 45`, `R² = 0` (не объясняет дисперсию). В комментариях поясните: почему `R² = 0` у baseline (предсказывает среднее); какие стратегии есть (`mean`, `median`, `quantile`); почему для MAE лучше `median` (устойчив к выбросам); как сравнить с моделью (`RMSE_model < RMSE_dummy`). Подсказка: `from sklearn.dummy import DummyRegressor; DummyRegressor(strategy="median")`.

**Задача 13 (`train_test_split`: параметры).** Разберите параметры: `test_size`, `train_size`, `stratify`, `shuffle`, `random_state`. Ожидаемый вывод: `test_size=0.2` → 400 из 2000; `stratify=y` — пропорция классов сохранена; `shuffle=False` — первые 80% train (плохо для неслучайных данных). В комментариях поясните: почему `shuffle=True` — дефолт; когда `shuffle=False` (временные ряды); почему `stratify` критичен для дисбаланса; что будет с `y_train.mean()` без `stratify` (может «уплыть» на 5–10%). Подсказка: `train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=42)`.

**Задача 14 (`KFold`).** Реализуйте `KFold(n_splits=5, shuffle=True, random_state=42)`. Ожидаемый вывод: 5 фолдов, train ~1600, test ~400 в каждом. В комментариях поясните: что такое `K-Fold` (делим на `K` частей, `K` раз обучаем на `K−1`, тестируем на 1); почему `shuffle=True` (без него — смещение по порядку); почему `K=5` или `10` — стандарт (`K=10` точнее, `K=5` быстрее); почему `KFold` **не подходит** для классификации с дисбалансом (нужен `StratifiedKFold`); как `KFold` связан с `train_test_split` (`KFold` = `K` split'ов). Подсказка: `from sklearn.model_selection import KFold; kf = KFold(n_splits=5, shuffle=True, random_state=42)`.

**Задача 15 (`StratifiedKFold`).** Сравните `KFold` и `StratifiedKFold` на дисбалансе 90/10. Ожидаемый вывод: `KFold` может дать фолд с 0 положительных; `StratifiedKFold` — всегда 10%. В комментариях поясните: что такое stratification (сохраняет пропорцию классов в каждом фолде); почему для регрессии — `KFold`, для классификации — `StratifiedKFold`; почему без stratification оценка может «скакать»; как `cross_val_score` выбирает CV по умолчанию (для классификатора — stratified). Подсказка: `from sklearn.model_selection import StratifiedKFold; cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.

**Задача 16 (`GroupKFold`).** Если в данных есть «группы» (например, один читатель даёт 5 выдач), обычная CV «протекает» (один и тот же читатель в train и test). Примените `GroupKFold(groups=reader_id)`. Ожидаемый вывод: читатель целиком в одном фолде. В комментариях поясните: что такое data leakage через группы; когда это критично (медицина — пациенты, NLP — документы, временные — пользователи); чем `GroupKFold` отличается от `StratifiedKFold` (первый — про группы, второй — про классы); что такое `StratifiedGroupKFold` (оба сразу). Подсказка: `from sklearn.model_selection import GroupKFold; GroupKFold(n_splits=5)`.

**Задача 17 (`TimeSeriesSplit`).** Для временных рядов обычная CV «протекает» (train после test). Используйте `TimeSeriesSplit`. Ожидаемый вывод: train — прошлое, test — будущее; 5 фолдов с растущим train. В комментариях поясните: почему обычная `KFold` ломает временные ряды (data leakage из будущего); как работает `TimeSeriesSplit` (расширяющееся окно); что такое `gap` (пропуск между train и test); когда `PurgedKFold` нужен (финансы). Подсказка: `from sklearn.model_selection import TimeSeriesSplit; TimeSeriesSplit(n_splits=5, gap=0)`.

**Задача 18 (`cross_val_score` + `cross_validate`).** Оцените модель через `cross_val_score` (только score) и `cross_validate` (score + время обучения + время предсказания). Ожидаемый вывод: `scores = [0.78, 0.81, 0.76, 0.79, 0.80]`, `mean = 0.79`, `std = 0.02`; `cross_validate` даёт ещё `fit_time` и `score_time`. В комментариях поясните: зачем кросс-валидация (оценка стабильнее, зависит от split); почему `std` важен (высокий std → модель нестабильна); чем `cross_validate` лучше `cross_val_score` (несколько метрик, время); как выбрать `scoring` (`"f1"`, `"roc_auc"`, `"neg_mean_squared_error"`). Подсказка: `from sklearn.model_selection import cross_validate; results = cross_validate(model, X, y, cv=5, scoring=["accuracy", "f1"], return_train_score=True)`.

---

## 🔵 Уровень 3. Classification metrics (19–28)

**Что изучается:** `accuracy`, `confusion_matrix`, `precision`, `recall`, `F1`, `classification_report`, `ROC-AUC`, `PR-AUC`, `log_loss`, `matthews_corrcoef`, `cohen_kappa`, threshold tuning.

**Задача 19 (`accuracy`: ловушка дисбаланса).** Обучите `LogisticRegression` и сравните с `DummyClassifier`. Ожидаемый вывод: `LogReg accuracy = 0.78`, `Dummy = 0.58` — модель лучше, но… **какой recall?** В комментариях поясните: почему accuracy обманчива при дисбалансе (модель «всегда 0» на 95/5 даёт 95%); почему на дисбалансе смотрят на `precision/recall/F1`; почему `accuracy` — плохая метрика для медицинских тестов (ложноотрицательные критичны); когда accuracy всё-таки ок (сбалансированные классы, равные ошибки). Подсказка: `from sklearn.metrics import accuracy_score`.

**Задача 20 (`confusion_matrix`).** Постройте матрицу ошибок для `LogisticRegression`. Ожидаемый вывод: 2×2 таблица `TN, FP, FN, TP`. В комментариях поясните: что означают `TN/FP/FN/TP`; чем `FP` (ложноположительный) отличается от `FN` (ложноотрицательный) по «цене»; как из матрицы получить `accuracy`, `precision`, `recall`, `F1`; почему визуализация через `ConfusionMatrixDisplay` нагляднее. Подсказка: `from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay; cm = confusion_matrix(y_test, y_pred)`.

**Задача 21 (`precision`, `recall`, `F1`).** Вычислите три метрики для `LogisticRegression`. Ожидаемый вывод: `precision = 0.72`, `recall = 0.65`, `F1 = 0.68`. В комментариях поясните: `precision = TP / (TP + FP)` — «из предсказанных 1 сколько верных»; `recall = TP / (TP + FN)` — «из реальных 1 сколько нашли»; `F1 = 2·P·R / (P + R)` — гармоническое среднее; когда важнее precision (спам-фильтр), когда recall (рак-скрининг); почему `F1` «наказывает» за дисбаланс между P и R. Подсказка: `from sklearn.metrics import precision_score, recall_score, f1_score`.

**Задача 22 (`classification_report`).** Выведите полный отчёт: precision/recall/F1 для **каждого** класса + `macro avg` / `weighted avg`. Ожидаемый вывод: таблица из 4 строк + support. В комментариях поясните: чем `macro avg` (среднее по классам, все равны) отличается от `weighted avg` (взвешенное по support); когда смотреть на какой (дисбаланс → `weighted` или `macro`); что такое `support` (число примеров класса в test); почему `classification_report` — стандарт отчёта для классификации. Подсказка: `from sklearn.metrics import classification_report; print(classification_report(y_test, y_pred))`.

**Задача 23 (`ROC-AUC` + `ROC-кривая`).** Постройте ROC-кривую и вычислите `AUC`. Ожидаемый вывод: `AUC = 0.82`; ROC-кривая выше диагонали. В комментариях поясните: что такое ROC (TPR vs FPR при разных threshold); что такое `AUC` (0.5 — случай, 1 — идеал); почему `AUC` — метрика **ранжирования**, не зависящая от threshold; когда `PR-AUC` лучше (сильный дисбаланс); как интерпретировать `AUC = 0.82` (модель ранжирует лучше, чем случай в 82% случаев). Подсказка: `from sklearn.metrics import roc_auc_score, roc_curve; RocCurveDisplay.from_estimator(model, X_test, y_test)`.

**Задача 24 (`PR-AUC`).** Постройте Precision-Recall-кривую и вычислите `average_precision_score`. Ожидаемый вывод: `PR-AUC = 0.65`; на дисбалансе 95/5 PR-AUC информативнее ROC-AUC. В комментариях поясните: когда PR-AUC лучше ROC-AUC (сильный дисбаланс, важен положительный класс); что такое `baseline` для PR-AUC (доля положительных, а не 0.5); почему ROC-AUC «завышен» на дисбалансе (FPR считается по огромному числу TN); как `PrecisionRecallDisplay` строит кривую. Подсказка: `from sklearn.metrics import average_precision_score, PrecisionRecallDisplay`.

**Задача 25 (`log_loss`).** Вычислите `log_loss` для вероятностей `predict_proba`. Ожидаемый вывод: `log_loss = 0.42`; ниже — лучше. В комментариях поясните: что такое log-loss (`−Σ y·log(p) + (1−y)·log(1−p)`); почему log-loss — «настоящая» метрика вероятностей (не только класса); когда log-loss важен (нужны калиброванные вероятности, а не только класс); почему accuracy — частный случай log-loss при threshold 0.5. Подсказка: `from sklearn.metrics import log_loss; log_loss(y_test, y_proba)`.

**Задача 26 (`matthews_corrcoef`).** Вычислите `MCC` — метрику, устойчивую к дисбалансу. Ожидаемый вывод: `MCC = 0.45`; диапазон `[−1, 1]`; `0` — случай. В комментариях поясните: почему MCC «лучше F1» на дисбалансе (учитывает все 4 ячейки); когда MCC предпочтителен (дисбаланс, разные цены ошибок); почему MCC не «популярен» (сложнее интерпретировать); как MCC связан с χ² (для 2×2 — пропорционален). Подсказка: `from sklearn.metrics import matthews_corrcoef`.

**Задача 27 (`cohen_kappa`).** Вычислите `Cohen's kappa` — согласие «с поправкой на случай». Ожидаемый вывод: `kappa = 0.42`; интерпретация: `0.4–0.6` — умеренное согласие. В комментариях поясните: что такое kappa (`(p_o − p_e) / (1 − p_e)`); почему accuracy может быть высокой «случайно» (при дисбалансе); когда kappa полезен (сравнение аннотаторов, мультикласс); почему kappa не «замена» F1. Подсказка: `from sklearn.metrics import cohen_kappa_score`.

**Задача 28 (threshold tuning).** Постройте `precision` и `recall` от threshold `0.1…0.9`. Ожидаемый вывод: threshold `0.3` → `precision = 0.55`, `recall = 0.85`; threshold `0.7` → `precision = 0.85`, `recall = 0.45`. В комментариях поясните: почему threshold `0.5` — «дефолт», но не «истина»; как выбрать threshold (бизнес-требования: «не пропустить должника» → низкий threshold; «не беспокоить зря» → высокий); что такое «threshold moving» (`precision_recall_curve`); почему на дисбалансе threshold tuning критичен; как сохранить threshold в `Pipeline`. Подсказка: `precision, recall, thresholds = precision_recall_curve(y_test, y_proba); f1_scores = 2 * precision * recall / (precision + recall); best_threshold = thresholds[np.argmax(f1_scores)]`.

---

## 🔴 Уровень 4. Regression metrics (29–33)

**Что изучается:** `MSE`, `RMSE`, `MAE`, `MAPE`, `R²`, `Adjusted R²`, `explained_variance`, `median_absolute_error`, `Huber`, интерпретация остатков.

**Задача 29 (`MSE`, `RMSE`, `MAE`).** Обучите `LinearRegression` на регрессии `fine` и вычислите три метрики. Ожидаемый вывод: `MSE = 1200`, `RMSE = 34.6`, `MAE = 22.1`. В комментариях поясните: MSE «наказывает» за большие ошибки (квадрат); RMSE в тех же единицах, что `y`; MAE устойчив к выбросам; когда какую метрику (RMSE — если большие ошибки критичны, MAE — если выбросы); почему `RMSE ≥ MAE` всегда. Подсказка: `from sklearn.metrics import mean_squared_error, mean_absolute_error; rmse = mean_squared_error(y, y_pred, squared=False)`.

**Задача 30 (`MAPE`).** Вычислите `MAPE` (средняя абсолютная процентная ошибка). Ожидаемый вывод: `MAPE = 18%`; проблема: при `y = 0` — деление на ноль. В комментариях поясните: что такое MAPE (`mean(|y − ŷ| / |y|) · 100%`); когда MAPE удобен (бизнесу понятно «ошибаемся на 18%»); почему MAPE ломается при `y = 0` или малых `y`; что такое `sMAPE` (симметричный) и когда он лучше. Подсказка: `from sklearn.metrics import mean_absolute_percentage_error`.

**Задача 31 (`R²`).** Вычислите `R²` для `LinearRegression` и `DummyRegressor`. Ожидаемый вывод: `R²(Linear) = 0.72`, `R²(Dummy) = 0`. В комментариях поясните: `R² = 1 − SS_res / SS_tot` — доля объяснённой дисперсии; почему `R²` может быть отрицательным (модель **хуже** среднего); почему `R² = 0.72` — «хорошо» для реальных данных, но «плохо» для физики; почему `R²` не говорит о **причинности**; когда `R²` вводит в заблуждение (нелинейные зависимости, выбросы). Подсказка: `from sklearn.metrics import r2_score`.

**Задача 32 (`Adjusted R²`).** Сравните `R²` и `Adjusted R²` для моделей с разным числом признаков. Ожидаемый вывод: `R²` растёт с числом признаков, `Adjusted R²` — штрафует. В комментариях поясните: формулу `Adj R² = 1 − (1−R²)(n−1)/(n−p−1)`; зачем (R² всегда растёт при добавлении признаков, даже случайных); когда `Adjusted R²` критичен (много признаков, сравнение моделей); почему sklearn **не** даёт `Adjusted R²` (считайте вручную). Подсказка: `adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)`.

**Задача 33 (остатки).** Постройте остатки модели: `residuals = y − ŷ`; графики `residuals vs ŷ` и QQ-plot. Ожидаемый вывод: остатки без паттерна, QQ-plot близок к прямой. В комментариях поясните: зачем анализировать остатки (проверка линейности, гомоскедастичности, нормальности); что такое `heteroscedasticity` («воронка» — разный разброс) и почему это проблема; как выглядит «плохой» график (кривая, воронка, кластеры); что делать при нарушении (лог-преобразование, нелинейная модель, `HuberRegressor`). Подсказка: `residuals = y_test - y_pred; plt.scatter(y_pred, residuals); sm.qqplot(residuals, line="45")`.

---

## 🟣 Уровень 5. Линейные модели (34–43)

**Что изучается:** `LinearRegression`, `Ridge`, `Lasso`, `ElasticNet`, `HuberRegressor`, `LogisticRegression`, `SGDClassifier`, регуляризация (`alpha`, `C`), интерпретация коэффициентов, `statsmodels` vs sklearn, мультиколлинеарность.

**Задача 34 (`LinearRegression`: база).** Обучите `LinearRegression` на регрессии `fine ~ days_overdue + days_held + reader_age`. Ожидаемый вывод: `coef = [48.2, 1.5, -0.3]`, `intercept = 12.5`, `R² = 0.72`. В комментариях поясните: что такое `coef` (изменение `y` при +1 к признаку, **при фиксированных остальных**); что такое `intercept` (значение `y` при всех `x = 0`, часто бессмысленно); почему `R²` не говорит о качестве на новых данных (см. переобучение); почему `LinearRegression` не любит мультиколлинеарность (коэффициенты «прыгают»). Подсказка: `from sklearn.linear_model import LinearRegression`.

**Задача 35 (`Ridge`: L2-регуляризация).** Обучите `Ridge` с `alpha = 0.1, 1, 10, 100`. Ожидаемый вывод: `alpha ↑ → coef ↓` (сжимаются к 0), `R²` слегка падает. В комментариях поясните: что такое L2-регуляризация (`+ α·Σβ²`); почему `Ridge` стабилизирует коэффициенты при мультиколлинеарности; почему `Ridge` **не** зануляет коэффициенты (только сжимает); как выбрать `alpha` (`RidgeCV` или `GridSearchCV`); почему `Ridge` — дефолт для регрессии. Подсказка: `from sklearn.linear_model import Ridge; Ridge(alpha=1.0)`.

**Задача 36 (`Lasso`: L1-регуляризация).** Обучите `Lasso` с разными `alpha`. Ожидаемый вывод: при большом `alpha` часть `coef = 0` — «отбор признаков». В комментариях поясните: что такое L1 (`+ α·Σ|β|`); почему L1 **зануляет** коэффициенты (геометрия ромба); когда `Lasso` лучше `Ridge` (много признаков, нужен отбор); чем `Lasso` плох при коррелированных признаках (выбирает «один из»); что такое `ElasticNet` (L1 + L2). Подсказка: `from sklearn.linear_model import Lasso; Lasso(alpha=0.1)`.

**Задача 37 (`ElasticNet`).** Сравните `Ridge`, `Lasso`, `ElasticNet` на данных с коррелированными признаками. Ожидаемый вывод: `ElasticNet` — компромисс: и отбор, и стабильность. В комментариях поясните: параметры `alpha` (сила) и `l1_ratio` (баланс L1/L2); почему `ElasticNet` — дефолт для многих задач; когда чистый `Lasso` или чистый `Ridge` лучше; как `l1_ratio` влияет на число ненулевых коэффициентов. Подсказка: `from sklearn.linear_model import ElasticNet; ElasticNet(alpha=0.1, l1_ratio=0.5)`.

**Задача 38 (`HuberRegressor`).** Обучите `HuberRegressor` на данных с выбросами (`fine = 99999`). Ожидаемый вывод: `RMSE(Huber) < RMSE(LinearRegression)`. В комментариях поясните: что такое Huber loss (`MSE` при малых ошибках, `MAE` при больших); почему `Huber` устойчив к выбросам; когда `Huber` лучше `Ridge` (выбросы в `y`); почему `Huber` медленнее (итеративная оптимизация). Подсказка: `from sklearn.linear_model import HuberRegressor; HuberRegressor(epsilon=1.35)`.

**Задача 39 (`LogisticRegression`: база).** Обучите `LogisticRegression` на классификации `returned_on_time`. Ожидаемый вывод: `coef`, `intercept`, `predict_proba`, `predict`. В комментариях поясните: чем `LogisticRegression` отличается от `LinearRegression` (логит-преобразование, `p ∈ [0, 1]`); что такое `predict_proba` (вероятность класса); как `coef` связан с `odds ratio` (`exp(coef)`); почему `LogisticRegression` — baseline для бинарной классификации. Подсказка: `from sklearn.linear_model import LogisticRegression; LogisticRegression(max_iter=1000)`.

**Задача 40 (регуляризация `LogisticRegression`: `C`).** Сравните `C = 0.01, 0.1, 1, 10, 100`. Ожидаемый вывод: `C ↓ → регуляризация ↑ → coef ↓`; `C → ∞` — почти без регуляризации. В комментариях поясните: почему `C` — **обратная** сила регуляризации (маленький `C` — сильная); чем `penalty="l1"` отличается от `"l2"` (отбор vs сжатие); почему `solver="liblinear"` нужен для `l1`; как выбрать `C` через `GridSearchCV`. Подсказка: `LogisticRegression(C=1.0, penalty="l2", solver="lbfgs")`.

**Задача 41 (`SGDClassifier`).** Обучите `SGDClassifier(loss="log_loss")` — «стохастический градиентный спуск». Ожидаемый вывод: `accuracy ≈ 0.77` (чуть хуже `LogisticRegression`), но **в 10–100 раз быстрее** на больших данных. В комментариях поясните: что такое SGD (обновление по мини-батчам); почему SGD быстрее на больших данных; что такое `learning_rate`, `alpha`, `max_iter`; когда SGD лучше `LogisticRegression` (big data, онлайн-обучение); почему SGD чувствителен к масштабу (стандартизация обязательна). Подсказка: `from sklearn.linear_model import SGDClassifier; SGDClassifier(loss="log_loss", max_iter=1000, tol=1e-3)`.

**Задача 42 (интерпретация коэффициентов).** Для `LogisticRegression` выведите коэффициенты и `odds ratio`. Ожидаемый вывод: `coef(days_held) = -0.08`, `OR = 0.92` («+1 день → шанс вернуть вовремя ×0.92»). В комментариях поясните: как интерпретировать `coef` (знак и «при прочих равных»); что такое `odds ratio` (`OR > 1` — увеличивает шанс); почему **нельзя** говорить «+1 день → −8% вероятность» (это не линейно); почему масштаб признаков важен (стандартизованные `coef` сравнимы, нет — нет). Подсказка: `np.exp(model.coef_[0])`.

**Задача 43 (`statsmodels` vs sklearn).** Постройте ту же `LogisticRegression` через `statsmodels` (`smf.logit`). Ожидаемый вывод: `coef` совпадают, но `statsmodels` даёт `std err`, `p-value`, `CI` — sklearn **не** даёт. В комментариях поясните: чем `statsmodels` отличается от sklearn (инференс vs предсказание); когда `statsmodels` (`p-value` для коэффициентов, ДИ, отчёты); когда sklearn (`predict`, `Pipeline`, production); почему sklearn не даёт `p-value` (фокус на предсказании, не на инференсе); что такое `quasi-separation` (логистическая регрессия «расходится»). Подсказка: `import statsmodels.formula.api as smf; smf.logit("returned_on_time ~ days_held + reader_age", data=df).fit().summary()`.

---

## 🟠 Уровень 6. Деревья (44–49)

**Что изучается:** `DecisionTreeClassifier`, `DecisionTreeRegressor`, `max_depth`, `min_samples_leaf`, `min_samples_split`, `criterion` (gini/entropy), overfitting, визуализация дерева.

**Задача 44 (`DecisionTreeClassifier`: база).** Обучите дерево решений на `returned_on_time`. Ожидаемый вывод: `depth = 8`, `accuracy_train = 0.95`, `accuracy_test = 0.72` — **переобучение**. В комментариях поясните: как дерево строит splits (жадный выбор по `gini`/`entropy`); почему без ограничений дерево «запоминает» train (каждый лист — один пример); что такое `max_depth`, `min_samples_leaf`, `min_samples_split` (регуляризация); почему деревья не любят выбросы (split «подстраивается»). Подсказка: `from sklearn.tree import DecisionTreeClassifier; DecisionTreeClassifier(max_depth=5)`.

**Задача 45 (`max_depth`: борьба с переобучением).** Обучите деревья с `max_depth = 2, 3, 5, 10, None`. Постройте график `train_accuracy` vs `test_accuracy`. Ожидаемый вывод: `depth=3` — оптимум; `depth=10` — переобучение; `depth=None` — сильное переобучение. В комментариях поясните: что такое `bias-variance tradeoff` (глубокое дерево — низкий bias, высокий variance); как найти оптимум (`validation_curve`); почему «идеальное train accuracy» — плохой знак; почему деревья — «weak learners» (по отдельности плохи, но в ансамбле — сильны). Подсказка: `for d in [2, 3, 5, 10, None]: ...`.

**Задача 46 (`min_samples_leaf`).** Сравните `min_samples_leaf = 1, 5, 20, 100`. Ожидаемый вывод: `min_samples_leaf=20` — оптимум; большие листья → недообучение. В комментариях поясните: что такое `min_samples_leaf` (минимум примеров в листе); почему он работает как регуляризация; как `min_samples_split` отличается; когда `min_samples_leaf` важнее `max_depth` (шумные данные, маленькие выборки); как комбинировать (`max_depth=5`, `min_samples_leaf=20`). Подсказка: `DecisionTreeClassifier(max_depth=5, min_samples_leaf=20)`.

**Задача 47 (`criterion`: gini vs entropy).** Обучите деревья с `criterion="gini"` и `"entropy"`. Ожидаемый вывод: почти одинаковый accuracy; `entropy` чуть медленнее. В комментариях поясните: что такое `gini` (мера «загрязнённости» узла) и `entropy` (мера неопределённости); почему на практике разница минимальна; когда `entropy` предпочтительнее (баланс классов); что такое `log_loss` как критерий (для вероятностей). Подсказка: `DecisionTreeClassifier(criterion="entropy")`.

**Задача 48 (`DecisionTreeRegressor`).** Обучите `DecisionTreeRegressor` на `fine`. Ожидаемый вывод: `RMSE = 28`, `R² = 0.68` — лучше `LinearRegression` на нелинейных данных. В комментариях поясните: чем `DecisionTreeRegressor` отличается от классификатора (критерий — MSE/MAE); почему дерево строит «ступенчатое» предсказание; почему деревья плохо экстраполируют (не могут выйти за диапазон `y` в train); когда деревья лучше линейных моделей (нелинейность, взаимодействия). Подсказка: `from sklearn.tree import DecisionTreeRegressor`.

**Задача 49 (визуализация дерева).** Визуализируйте дерево с `max_depth=3` через `plot_tree`. Ожидаемый вывод: схема со сплитами, `gini`/`entropy`, `samples`, `value`. В комментариях поясните: как читать дерево (узел — сплит по признаку, лист — предсказание); что показывает `value` (распределение классов); зачем визуализировать (объяснимость для бизнеса); почему глубокое дерево «нечитаемо»; как экспортировать (`export_text`, `graphviz`). Подсказка: `from sklearn.tree import plot_tree; plot_tree(model, max_depth=3, feature_names=..., class_names=..., filled=True)`.

---

## 🟠 Уровень 7. Ансамбли (50–59)

**Что изучается:** `RandomForest`, `ExtraTrees`, `BaggingClassifier`, `GradientBoosting`, `HistGradientBoosting`, `AdaBoost`, `VotingClassifier`, `StackingClassifier`, `feature_importances_`.

**Задача 50 (`RandomForestClassifier`: база).** Обучите `RandomForest` с `n_estimators=100`. Ожидаемый вывод: `accuracy_train = 0.98`, `accuracy_test = 0.81` — **лучше** одного дерева. В комментариях поясните: как работает `RandomForest` (много деревьев на bootstrap-выборках + случайные признаки); почему ансамбль лучше одного дерева (bias не растёт, variance падает); зачем `n_estimators` (больше — стабильнее, но медленнее); почему `RandomForest` почти не переобучается (можно `max_depth=None`); когда `RandomForest` — хороший дефолт. Подсказка: `from sklearn.ensemble import RandomForestClassifier; RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)`.

**Задача 51 (`ExtraTreesClassifier`).** Сравните `RandomForest` и `ExtraTrees`. Ожидаемый вывод: `ExtraTrees` чуть быстрее, accuracy примерно та же. В комментариях поясните: чем `ExtraTrees` отличается (случайные thresholds вместо оптимальных); почему быстрее (меньше вычислений); почему может быть чуть менее точным (больше randomness); когда `ExtraTrees` оправдан (большие данные). Подсказка: `from sklearn.ensemble import ExtraTreesClassifier`.

**Задача 52 (`BaggingClassifier`).** Обучите `BaggingClassifier(base_estimator=DecisionTreeClassifier())`. Ожидаемый вывод: accuracy выше одного дерева, но ниже `RandomForest`. В комментариях поясните: что такое bagging (bootstrap + агрегация); чем bagging отличается от `RandomForest` (последний добавляет случайные признаки); когда bagging полезен (любая модель как base); что такое `oob_score` (out-of-bag — бесплатная CV). Подсказка: `from sklearn.ensemble import BaggingClassifier; BaggingClassifier(base_estimator=DecisionTreeClassifier(), n_estimators=100)`.

**Задача 53 (`GradientBoostingClassifier`).** Обучите `GradientBoostingClassifier`. Ожидаемый вывод: `accuracy_test = 0.83` — обычно **лучше** `RandomForest` на табличных данных. В комментариях поясните: как работает boosting (последовательно, каждое дерево исправляет ошибки предыдущего); чем boosting отличается от bagging (`RandomForest`); почему boosting переобучается легче (`learning_rate` и `n_estimators` критичны); что такое `early_stopping` (остановка по validation). Подсказка: `from sklearn.ensemble import GradientBoostingClassifier; GradientBoostingClassifier(n_estimators=100, learning_rate=0.1)`.

**Задача 54 (`learning_rate` в boosting).** Сравните `learning_rate = 0.01, 0.05, 0.1, 0.5` при `n_estimators = 500` (с `early_stopping`). Ожидаемый вывод: маленький `learning_rate` → нужно больше деревьев; оптимально `0.05–0.1`. В комментариях поясните: что такое `learning_rate` (вклад каждого дерева); почему «маленький learning_rate + много деревьев» часто лучше; почему это дороже; как `early_stopping_rounds` спасает от переобучения. Подсказка: `GradientBoostingClassifier(n_estimators=500, learning_rate=0.05, validation_fraction=0.1, n_iter_no_change=10)`.

**Задача 55 (`HistGradientBoosting`).** Обучите `HistGradientBoostingClassifier` — современный быстрый boosting. Ожидаемый вывод: `accuracy ≈ 0.84`, время обучения **в 10–50 раз меньше** `GradientBoosting`. В комментариях поясните: чем `HistGradientBoosting` отличается (`binned` признаки, `O(n)` вместо `O(n log n)`); почему он «дефолт» для табличных данных (быстрый, точный, поддерживает `NaN`); когда `XGBoost`/`LightGBM` всё равно лучше (экстремальные задачи, GPU); почему `HistGradientBoosting` поддерживает категориальные признаки «из коробки» (`categorical_features`). Подсказка: `from sklearn.ensemble import HistGradientBoostingClassifier`.

**Задача 56 (`AdaBoost`).** Обучите `AdaBoostClassifier`. Ожидаемый вывод: accuracy ≈ `RandomForest`, но чувствителен к шуму. В комментариях поясните: как работает AdaBoost (перевес «трудных» примеров); чем отличается от Gradient Boosting (AdaBoost — про веса, GB — про градиенты); почему AdaBoost чувствителен к выбросам (они получают огромный вес); когда AdaBoost всё ещё полезен (простые задачи, интерпретируемость). Подсказка: `from sklearn.ensemble import AdaBoostClassifier; AdaBoostClassifier(n_estimators=100)`.

**Задача 57 (`VotingClassifier`).** Соберите `VotingClassifier(estimators=[LogReg, RF, SVC], voting="soft")`. Ожидаемый вывод: accuracy чуть выше лучшей модели. В комментариях поясните: что такое voting (hard — по классам, soft — по вероятностям); почему soft обычно лучше (использует `predict_proba`); почему voting помогает (разные модели ошибаются по-разному); когда voting — плохая идея (коррелированные модели). Подсказка: `from sklearn.ensemble import VotingClassifier; VotingClassifier(estimators=[("lr", LogReg), ("rf", RF), ("svc", SVC(probability=True))], voting="soft")`.

**Задача 58 (`StackingClassifier`).** Соберите `StackingClassifier` с meta-моделью `LogisticRegression`. Ожидаемый вывод: accuracy выше voting, но обучение дороже. В комментариях поясните: что такое stacking (meta-модель учится на предсказаниях базовых моделей); чем отличается от voting (meta-модель ≠ голосование); что такое `passthrough` (добавить исходные признаки в meta); почему stacking может переобучиться (нужна CV внутри); когда stacking оправдан (Kaggle, максимум accuracy). Подсказка: `from sklearn.ensemble import StackingClassifier; StackingClassifier(estimators=[...], final_estimator=LogisticRegression())`.

**Задача 59 (`feature_importances_`).** Для `RandomForest` выведите `feature_importances_`. Ожидаемый вывод: `days_held = 0.35`, `days_overdue = 0.28`, `genre = 0.12`, … В комментариях поясните: что такое feature importance (средний вклад в снижение impurity); почему **встроенная** importance смещена (завышает high-cardinality признаки); почему `permutation_importance` надёжнее (задача 91); как визуализировать (`barh`); почему «важность» ≠ «причинность». Подсказка: `rf.feature_importances_`.

---

## 🟠 Уровень 8. KNN, SVM, Naive Bayes (60–65)

**Что изучается:** `KNeighborsClassifier`, `KNeighborsRegressor`, `SVC`, `SVR`, `GaussianNB`, `MultinomialNB`, выбор `k`, `kernel`, `C`, `gamma`, `epsilon`.

**Задача 60 (`KNeighborsClassifier`: база).** Обучите KNN с `n_neighbors=5` на стандартизованных признаках. Ожидаемый вывод: `accuracy = 0.74`. В комментариях поясните: как работает KNN (голосование `k` ближайших соседей); почему **обязательна** стандартизация (расстояние зависит от масштаба); почему KNN медленный на test (`O(n·d)` на каждый пример); почему `k` — гиперпараметр (маленький `k` — переобучение, большой — недообучение). Подсказка: `from sklearn.neighbors import KNeighborsClassifier; KNeighborsClassifier(n_neighbors=5)`.

**Задача 61 (выбор `k`).** Переберите `k = 1, 3, 5, 9, 15, 25, 50` и постройте `train/test accuracy`. Ожидаемый вывод: оптимум `k ≈ 9`; `k=1` — переобучение; `k=50` — недообучение. В комментариях поясните: почему `k=1` даёт `train accuracy = 1.0` (запоминает каждый пример); почему большой `k` «размывает» границы; как выбрать `k` (`GridSearchCV`); почему `k` нечётный (избежать ничьей в бинарной классификации). Подсказка: `for k in [1, 3, 5, 9, 15, 25, 50]: ...`.

**Задача 62 (`SVC`: линейный и RBF kernel).** Обучите `SVC(kernel="linear")` и `SVC(kernel="rbf")`. Ожидаемый вывод: `linear accuracy = 0.76`, `rbf = 0.80`. В комментариях поясните: что такое SVM (максимальный margin); что такое support vectors; чем `linear` отличается от `rbf` (линейная vs нелинейная граница); как `C` и `gamma` влияют на переобучение; почему SVM не масштабируется на большие данные (`O(n²)`). Подсказка: `from sklearn.svm import SVC; SVC(kernel="rbf", C=1.0, gamma="scale")`.

**Задача 63 (`SVR`: регрессия).** Обучите `SVR` на регрессии `fine`. Ожидаемый вывод: `RMSE = 30`, `MAE = 20`. В комментариях поясните: чем `SVR` отличается от `SVC` (непрерывный target, epsilon-tube); что такое `epsilon` (ширина «трубы», внутри которой ошибка = 0); когда `SVR` лучше `LinearRegression` (нелинейность); почему `SVR` медленный на больших данных. Подсказка: `from sklearn.svm import SVR; SVR(kernel="rbf", C=1.0, epsilon=0.1)`.

**Задача 64 (`GaussianNB`).** Обучите `GaussianNB` на классификации. Ожидаемый вывод: `accuracy = 0.72`, обучение **мгновенное**. В комментариях поясните: как работает Naive Bayes (теорема Байеса + «наивное» предположение независимости); почему «наивное» предположение редко верно, но всё равно работает; почему `GaussianNB` для непрерывных признаков, `MultinomialNB` — для текста; когда NB — хороший baseline (текст, маленькие данные); почему NB «недооценивает» вероятности (плохо калиброван). Подсказка: `from sklearn.naive_bayes import GaussianNB; GaussianNB()`.

**Задача 65 (`MultinomialNB`: текст).** Сгенерируйте текст-данные (`CountVectorizer` + `MultinomialNB`) — например, жанр по описанию. Ожидаемый вывод: `accuracy = 0.85` на тексте. В комментариях поясните: что делает `CountVectorizer` (bag-of-words); почему `MultinomialNB` — стандарт для текста; что такое `alpha` (сглаживание Лапласа, чтобы не было `0` вероятностей); как `TfidfVectorizer` улучшает; когда текст → NB, когда → `LogisticRegression`/трансформеры. Подсказка: `from sklearn.feature_extraction.text import CountVectorizer; from sklearn.naive_bayes import MultinomialNB`.

---

## 🟣 Уровень 9. Кросс-валидация и тюнинг (66–73)

**Что изучается:** `GridSearchCV`, `RandomizedSearchCV`, `HalvingGridSearchCV`, `learning_curve`, `validation_curve`, вложенная CV, `Pipeline` + `GridSearchCV`, `optuna` (обзорно).

**Задача 66 (`GridSearchCV`: база).** Переберите `C` для `LogisticRegression`: `[0.01, 0.1, 1, 10, 100]` через `GridSearchCV(cv=5)`. Ожидаемый вывод: `best_params = {"C": 1}`, `best_score = 0.80`. В комментариях поясните: что такое `GridSearchCV` (полный перебор + CV); почему это дорого (`n_params × n_folds × время обучения`); почему `best_score_` — CV-оценка, не test; почему `refit=True` автоматически обучает лучшую модель на всём train; как `n_jobs=-1` ускоряет. Подсказка: `from sklearn.model_selection import GridSearchCV; GridSearchCV(model, param_grid, cv=5, n_jobs=-1)`.

**Задача 67 (`RandomizedSearchCV`).** Переберите `C` и `penalty` для `LogisticRegression` через `RandomizedSearchCV(n_iter=20)`. Ожидаемый вывод: `best_params` примерно как у `GridSearchCV`, но **в 5–10 раз быстрее**. В комментариях поясните: чем `RandomizedSearchCV` отличается (случайная выборка из `param_distributions`); почему он часто лучше `GridSearchCV` (не тратит время на «бесполезные» комбинации); когда `Randomized` — дефолт (много гиперпараметров, дорогое обучение); как `n_iter` связан с покрытием пространства. Подсказка: `from sklearn.model_selection import RandomizedSearchCV; RandomizedSearchCV(model, param_distributions, n_iter=20, cv=5)`.

**Задача 68 (`HalvingGridSearchCV`).** Используйте `HalvingGridSearchCV` (successive halving) для быстрого поиска. Ожидаемый вывод: тот же `best_params`, **в 3–10 раз быстрее** `GridSearchCV`. В комментариях поясните: как работает successive halving (обучает много моделей на малых данных, оставляет лучшие, увеличивает данные); почему это новее `GridSearchCV` (scikit-learn 0.24+); когда `HalvingGridSearchCV` — лучший выбор (большое пространство гиперпараметров); какие ограничения (не всё поддерживает). Подсказка: `from sklearn.experimental import enable_halving_search_cv; from sklearn.model_selection import HalvingGridSearchCV`.

**Задача 69 (`learning_curve`).** Постройте `learning_curve`: accuracy от размера train. Ожидаемый вывод: кривая train падает, test растёт; разрыв → переобучение или недообучение. В комментариях поясните: как читать `learning_curve` (большой разрыв — переобучение, обе низко — недообучение, обе сходятся — оптимум); почему «больше данных» не всегда помогает (если модель недообучена); как использовать для решения «собирать ли ещё данные»; почему `learning_curve` дорогая (обучает `n_points × n_folds` моделей). Подсказка: `from sklearn.model_selection import learning_curve; train_sizes, train_scores, test_scores = learning_curve(model, X, y, cv=5)`.

**Задача 70 (`validation_curve`).** Постройте `validation_curve` для `max_depth` дерева. Ожидаемый вывод: train accuracy растёт с `max_depth`, test — «горб» (оптимум). В комментариях поясните: чем `validation_curve` отличается от `learning_curve` (гиперпараметр vs размер train); как найти оптимум; почему test-кривая «падает» после оптимума (переобучение); как `validation_curve` связана с `GridSearchCV` (одно и то же, но визуализация). Подсказка: `from sklearn.model_selection import validation_curve; train_scores, test_scores = validation_curve(model, X, y, param_name="max_depth", param_range=[2, 3, 5, 10], cv=5)`.

**Задача 71 (вложенная CV).** Покажите, почему `best_score_` из `GridSearchCV` **оптимистичен** (тюнинг на тех же фолдах). Реализуйте вложенную CV: внешняя CV для оценки, внутренняя — для тюнинга. Ожидаемый вывод: `nested_score < best_score_` (обычно на 1–3%). В комментариях поясните: что такое «optimization bias» (тюнинг «подглядывает» в test); почему `best_score_` нельзя использовать как честную оценку; как `nested CV` решает (`GridSearchCV` внутри `cross_val_score`); почему в реальных проектах достаточно hold-out test (если он не трогался). Подсказка: `from sklearn.model_selection import cross_val_score; nested_scores = cross_val_score(GridSearchCV(model, params, cv=5), X, y, cv=5)`.

**Задача 72 (`Pipeline` + `GridSearchCV`).** Настройте гиперпараметры **внутри** `Pipeline` (например, `model__C`, `model__penalty`). Ожидаемый вывод: `best_params = {"model__C": 1, "model__penalty": "l2"}`. В комментариях поясните: почему `Pipeline` + `GridSearchCV` — единственный «честный» способ (препроцессинг внутри CV, нет leakage); как работают имена параметров (`step__param`); почему нельзя тюнить препроцессинг **вне** `Pipeline` (data leakage); как передать `param_grid` с разными префиксами. Подсказка: `param_grid = {"model__C": [0.1, 1, 10], "prep__num__with_mean": [True, False]}`.

**Задача 73 (`optuna`: обзорно).** Используйте `optuna` (байесовская оптимизация) для `HistGradientBoosting`. Ожидаемый вывод: `best_params` находятся быстрее и часто лучше `RandomizedSearchCV`. В комментариях поясните: чем `optuna` лучше (байесовская оптимизация — «учится» на результатах); что такое `TPE` (Tree-structured Parzen Estimator); когда optuna оправдана (дорогие модели, мало времени); какие альтернативы (`hyperopt`, `nevergrad`). Подсказка: `import optuna; study = optuna.create_study(direction="maximize"); study.optimize(objective, n_trials=50)`.

---

## 🔵 Уровень 10. Кластеризация (74–79)

**Что изучается:** `KMeans`, `MiniBatchKMeans`, `DBSCAN`, `AgglomerativeClustering`, `silhouette_score`, `calinski_harabasz_score`, `davies_bouldin_score`, elbow method, интерпретация кластеров.

**Задача 74 (`KMeans`: база).** Кластеризуйте читателей по поведению (`days_held`, `days_overdue`, `fine`) на `k=3`. Ожидаемый вывод: 3 кластера; `cluster_centers_` показывает «профили». В комментариях поясните: как работает `KMeans` (минимизация внутрикластерного расстояния); почему нужно стандартизовать (расстояние зависит от масштаба); почему `KMeans` чувствителен к инициализации (`n_init=10`); почему результат зависит от `random_state`; чем кластеры отличаются от классов (нет истины). Подсказка: `from sklearn.cluster import KMeans; KMeans(n_clusters=3, n_init=10, random_state=42)`.

**Задача 75 (elbow method).** Постройте `inertia` (within-cluster sum of squares) для `k = 1…10`. Ожидаемый вывод: «локоть» на `k=3` или `k=4`. В комментариях поясните: что такое `inertia`; почему она всегда падает с ростом `k` (при `k=n` — 0); как найти «локоть» (визуально или `knee`-алгоритм); почему elbow — эвристика (не всегда чёткий); что делать, если локоть не виден (`silhouette`). Подсказка: `inertias = [KMeans(n_clusters=k, n_init=10).fit(X).inertia_ for k in range(1, 11)]`.

**Задача 76 (`silhouette_score`).** Вычислите `silhouette_score` для `k = 2…10`. Ожидаемый вывод: максимум на `k=3` (`silhouette ≈ 0.55`). В комментариях поясните: что такое silhouette (`(b − a) / max(a, b)`, где `a` — внутрикластерное расстояние, `b` — до ближайшего другого); диапазон `[−1, 1]`; почему `silhouette` лучше elbow (количественная метрика); почему `silhouette` дорогой (`O(n²)`); как `silhouette_samples` помогает найти «плохие» точки. Подсказка: `from sklearn.metrics import silhouette_score; silhouette_score(X, labels)`.

**Задача 77 (`DBSCAN`).** Кластеризуйте через `DBSCAN(eps=0.5, min_samples=5)`. Ожидаемый вывод: найдено 2–4 кластера + шум (`label = -1`). В комментариях поясните: как работает `DBSCAN` (плотность: `eps`-окрестность, `min_samples`); почему `DBSCAN` находит кластеры **любой формы** (в отличие от `KMeans`); как выбирать `eps` (`k-distance plot`); почему `DBSCAN` помечает выбросы как `-1`; когда `DBSCAN` лучше `KMeans` (неизвестное число кластеров, шум, нелинейные формы). Подсказка: `from sklearn.cluster import DBSCAN; DBSCAN(eps=0.5, min_samples=5)`.

**Задача 78 (`AgglomerativeClustering`).** Кластеризуйте через иерархическую кластеризацию. Ожидаемый вывод: дендрограмма + кластеры. В комментариях поясните: как работает agglomerative (снизу вверх, объединяет ближайшие); что такое linkage (`ward`, `complete`, `average`, `single`); как читать дендрограмму; когда иерархическая кластеризация лучше `KMeans` (нужна иерархия, маленькие данные); почему `O(n³)` — медленно. Подсказка: `from sklearn.cluster import AgglomerativeClustering; from scipy.cluster.hierarchy import dendrogram, linkage`.

**Задача 79 (интерпретация кластеров).** Для каждого кластера выведите `mean` по каждому признаку. Ожидаемый вывод: кластер 0 — «должники», кластер 1 — «аккуратные», кластер 2 — «новички». В комментариях поясните: зачем интерпретировать кластеры (бизнесу нужны «профили»); как назвать кластеры (по средним значениям); что такое «профилирование»; почему кластеры — «гипотеза», а не «истина»; как использовать для сегментации (маркетинг, таргетинг). Подсказка: `df_clustered.groupby("cluster").mean()`.

---

## 🟣 Уровень 11. Снижение размерности (80–84)

**Что изучается:** `PCA`, `TruncatedSVD`, `TSNE`, `UMAP` (обзорно), выбор числа компонент, `explained_variance_ratio_`, `PCA` как препроцессинг.

**Задача 80 (`PCA`: база).** Примените `PCA(n_components=2)` к 10 признакам. Ожидаемый вывод: `explained_variance_ratio_ = [0.35, 0.22]` (58% дисперсии в 2 компонентах). В комментариях поясните: что делает PCA (линейное преобразование в ортогональные компоненты); что такое `explained_variance_ratio_`; почему PCA **не** учитывает `y` (unsupervised); когда PCA полезен (визуализация, снижение размерности, шум); почему масштаб важен (стандартизация обязательна); почему интерпретация компонент сложна. Подсказка: `from sklearn.decomposition import PCA; pca = PCA(n_components=2); X_pca = pca.fit_transform(X_scaled)`.

**Задача 81 (выбор числа компонент PCA).** Постройте график `explained_variance_ratio_` для всех компонент. Ожидаемый вывод: «локоть» на 3–4 компонентах; 95% дисперсии — 5 компонент. В комментариях поясните: как выбрать `n_components` (95% дисперсии, `n_components=0.95`); почему «оставить 2 для визуализации» — не то же, что «2 достаточно для модели»; когда PCA **вредит** (потеря интерпретируемости, линейность); почему `PCA` + `LogisticRegression` может работать лучше, чем `LogisticRegression` на оригинале (шумоподавление). Подсказка: `pca = PCA().fit(X_scaled); plt.plot(np.cumsum(pca.explained_variance_ratio_))`.

**Задача 82 (`TruncatedSVD`).** Примените `TruncatedSVD` к разреженной матрице (например, one-hot с 1000+ категорий). Ожидаемый вывод: `TruncatedSVD` работает с разреженными данными, `PCA` — нет. В комментариях поясните: чем `TruncatedSVD` отличается от `PCA` (не центрирует данные, работает с `sparse`); когда использовать (`CountVectorizer` — текст, one-hot high-cardinality); что такое `LSA` (latent semantic analysis) — это `TruncatedSVD` на TF-IDF; почему PCA не работает с `sparse` (центрирование делает матрицу плотной). Подсказка: `from sklearn.decomposition import TruncatedSVD`.

**Задача 83 (`TSNE`: визуализация).** Примените `TSNE(n_components=2)` к тем же данным. Ожидаемый вывод: 2D-карта, где похожие точки рядом. В комментариях поясните: чем `TSNE` отличается от PCA (нелинейное, сохраняет локальные связи); почему `TSNE` **только** для визуализации (не для модели); что такое `perplexity` (~5–50); почему `TSNE` медленный (`O(n²)`); почему результат зависит от `random_state`; когда `UMAP` лучше (быстрее, сохраняет глобальную структуру). Подсказка: `from sklearn.manifold import TSNE; TSNE(n_components=2, perplexity=30, random_state=42)`.

**Задача 84 (`PCA` как препроцессинг).** Сравните `LogisticRegression` на оригинале и на `PCA(0.95)` внутри `Pipeline`. Ожидаемый вывод: accuracy примерно та же, обучение быстрее, интерпретируемость ниже. В комментариях поясните: когда PCA как препроцессинг полезен (много признаков, коррелированные); когда **вредит** (теряется интерпретация, `coef` нечитаем); как `PCA` в `Pipeline` защищает от leakage (fit только на train); что такое `KernelPCA` (нелинейное снижение размерности). Подсказка: `Pipeline([("scaler", StandardScaler()), ("pca", PCA(0.95)), ("model", LogisticRegression())])`.

---

## 🟠 Уровень 12. Feature engineering & selection (85–94)

**Что изучается:** `VarianceThreshold`, `SelectKBest`, `mutual_info_classif`, `f_classif`, `RFE`, `RFECV`, `SelectFromModel`, `TargetEncoder`, `PolynomialFeatures` + `SelectKBest`, feature engineering (даты, агрегаты, взаимодействия).

**Задача 85 (`VarianceThreshold`).** Удалите константные и почти константные признаки. Ожидаемый вывод: удалено 2 признака (`variance < 0.01`). В комментариях поясните: зачем удалять константы (не несут информации, замедляют модель); что делает `VarianceThreshold`; почему для бинарных признаков порог `0.8·p·(1−p)`; когда это первый шаг любого пайплайна. Подсказка: `from sklearn.feature_selection import VarianceThreshold; VarianceThreshold(threshold=0.01)`.

**Задача 86 (`SelectKBest` + `mutual_info_classif`).** Отберите топ-5 признаков по mutual information. Ожидаемый вывод: `selected = ["days_held", "days_overdue", ...]`. В комментариях поясните: что такое mutual information (сколько информации признак даёт о `y`); почему MI ловит **нелинейные** связи (в отличие от корреляции); когда `f_classif` (ANOVA F) лучше; почему `SelectKBest` — быстрый и простой baseline. Подсказка: `from sklearn.feature_selection import SelectKBest, mutual_info_classif; SelectKBest(mutual_info_classif, k=5)`.

**Задача 87 (`RFE`).** Отберите признаки через `RFE` с `LogisticRegression`. Ожидаемый вывод: топ-3 признака, остальные удалены. В комментариях поясните: как работает RFE (обучает модель, удаляет наименее важный, повторяет); почему RFE дороже `SelectKBest` (`n_features` обучений); когда RFE лучше (модель даёт `feature_importances_`); что такое `step` (сколько удалять за раз). Подсказка: `from sklearn.feature_selection import RFE; RFE(LogisticRegression(), n_features_to_select=3)`.

**Задача 88 (`RFECV`).** Используйте `RFECV` — RFE, который сам выбирает число признаков через CV. Ожидаемый вывод: `optimal_n_features = 5`, `cv_score = 0.81`. В комментариях поясните: чем `RFECV` отличается от `RFE` (автоматический выбор через CV); почему это дороже (CV внутри RFE); когда `RFECV` оправдан (мало признаков, дорогая модель); почему `RFECV` может переобучиться (CV используется и для отбора, и для оценки). Подсказка: `from sklearn.feature_selection import RFECV; RFECV(LogisticRegression(), cv=5)`.

**Задача 89 (`SelectFromModel`).** Отберите признаки по `feature_importances_` из `RandomForest`. Ожидаемый вывод: топ-5 признаков по важности. В комментариях поясните: как работает `SelectFromModel` (берёт важность из модели); почему быстрее `RFE`; когда `SelectFromModel` лучше `SelectKBest` (модель «знает» про взаимодействия); что такое `threshold` (`"mean"`, `"median"`, число). Подсказка: `from sklearn.feature_selection import SelectFromModel; SelectFromModel(RandomForestClassifier(n_estimators=100), threshold="mean")`.

**Задача 90 (`PolynomialFeatures` + `SelectKBest`).** Сгенерируйте 20 полиномиальных признаков, отберите топ-5. Ожидаемый вывод: `5 из 20` остались; accuracy выросла на 2–3%. В комментариях поясните: зачем полиномы (нелинейность для линейных моделей); почему после полиномов нужен отбор (иначе `p >> n`); почему `PolynomialFeatures(degree=3)` — уже 100+ признаков; когда полиномы **вредят** (переобучение, шум). Подсказка: `Pipeline([("poly", PolynomialFeatures(degree=2)), ("select", SelectKBest(f_regression, k=5)), ("model", Ridge())])`.

**Задача 91 (`permutation_importance`).** Вычислите `permutation_importance` для `RandomForest` на test. Ожидаемый вывод: `days_held: 0.08`, `days_overdue: 0.05`, `genre: 0.01`. В комментариях поясните: как работает `permutation_importance` (перемешивает признак → измеряет падение метрики); почему он честнее `feature_importances_` (не зависит от модели); почему считается на **test**, не на train; как `n_repeats` влияет на стабильность; почему `importance > 0` не значит «причина». Подсказка: `from sklearn.inspection import permutation_importance; result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)`.

**Задача 92 (`TargetEncoder`).** Признак `reader_id` (2000 уникальных) — one-hot даст 2000 столбцов. Используйте `TargetEncoder` (или `OrdinalEncoder` + `Ridge`). Ожидаемый вывод: 1 столбец вместо 2000; accuracy не падает. В комментариях поясните: что такое target encoding (замена категории средним `y`); почему нужна регуляризация (иначе leakage на train); как `TargetEncoder` в sklearn делает это через CV; когда `TargetEncoder` лучше one-hot (high cardinality); почему это риск утечки данных (нужен `fit` только на train). Подсказка: `from sklearn.preprocessing import TargetEncoder; TargetEncoder(smooth="auto")` (sklearn 1.3+).

**Задача 93 (feature engineering: даты).** Из `loan_date` извлеките `year, month, day_of_week, is_weekend`. Ожидаемый вывод: 4 новых признака; `month` и `day_of_week` — важные. В комментариях поясните: почему «сырая дата» бесполезна для модели; какие признаки из даты полезны (сезонность, день недели); что такое циклические признаки (`sin/cos` для месяца); как `pandas.dt` accessor упрощает; почему `is_weekend` может быть сильнее `day_of_week` (меньше категорий). Подсказка: `df["loan_date"] = pd.to_datetime(df["loan_date"]); df["month"] = df["loan_date"].dt.month`.

**Задача 94 (feature engineering: агрегаты + взаимодействия).** Добавьте: агрегаты по читателю (`reader_total_loans`, `reader_avg_fine`); взаимодействия (`days_held × book_price`, `days_overdue / days_held`). Ожидаемый вывод: 4 новых признака; accuracy выросла на 3–5%. В комментариях поясните: что такое «агрегированные признаки» (groupby + transform, только на train); почему они часто важнее «сырых»; что такое «взаимодействия» (произведение/отношение); как создать без leakage; почему агрегаты «вредят» при маленькой выборке на группу. Подсказка: `df["reader_total_loans"] = df.groupby("reader_id")["loan_id"].transform("count"); df["days_held_x_price"] = df["days_held"] * df["book_price"]`.

---

## 🔴 Уровень 13. Imbalanced + calibration (95–100)

**Что изучается:** дисбаланс классов, `class_weight="balanced"`, `SMOTE`, `RandomOverSampler`, `RandomUnderSampler`, `calibration_curve`, `CalibratedClassifierCV`, `isotonic`/`sigmoid`.

**Задача 95 (дисбаланс: `class_weight`).** На данных с дисбалансом 90/10 обучите `LogisticRegression(class_weight="balanced")` и сравните с обычной. Ожидаемый вывод: `class_weight="balanced"` повышает recall на меньшем классе (ценой precision). В комментариях поясните: что делает `class_weight` (веса классов обратно пропорциональны частоте); почему accuracy почти не меняется, но recall растёт; когда `class_weight` лучше `SMOTE` (проще, нет синтетики); почему `balanced` — не «серебряная пуля» (precision падает); как выбрать между `balanced` и threshold tuning. Подсказка: `LogisticRegression(class_weight="balanced")`.

**Задача 96 (`SMOTE`).** Примените `SMOTE` к train (только train, не test!). Ожидаемый вывод: баланс 50/50 после `SMOTE`; recall растёт, precision падает. В комментариях поясните: что делает `SMOTE` (синтезирует новые примеры меньшинства между соседями); почему **нельзя** применять к test (это leakage); почему `SMOTE` внутри `Pipeline` обязателен; когда `SMOTE` лучше `class_weight` (сильный дисбаланс, много данных); чем `SMOTE` плох (может «создать» шум, не работает с категориями). Подсказка: `from imblearn.over_sampling import SMOTE; from imblearn.pipeline import Pipeline; Pipeline([("smote", SMOTE()), ("model", LogisticRegression())])`.

**Задача 97 (`RandomUnderSampler`).** Примените `RandomUnderSampler` к train. Ожидаемый вывод: баланс 50/50, но **меньше данных** — accuracy может упасть. В комментариях поясните: что делает undersampling (удаляет большинство); почему быстрее `SMOTE`; когда undersampling лучше (много данных, дорогое обучение); почему теряется информация; что такое `NearMiss` (умный undersampling). Подсказка: `from imblearn.under_sampling import RandomUnderSampler`.

**Задача 98 (`calibration_curve`).** Проверьте калибровку вероятностей `RandomForest`. Ожидаемый вывод: `calibration_curve` — модель «переуверена» (`p = 0.9` → реально 0.7). В комментариях поясните: что такое калибровка (предсказанная вероятность = реальная частота); почему `RandomForest`/boosting плохо калиброваны; как исправить (`CalibratedClassifierCV`, `isotonic`, `sigmoid`); когда калибровка критична (медицина, финансы — важна вероятность, не только класс). Подсказка: `from sklearn.calibration import calibration_curve, CalibratedClassifierCV`.

**Задача 99 (`CalibratedClassifierCV`).** Оберните `RandomForest` в `CalibratedClassifierCV` с `method="isotonic"`. Ожидаемый вывод: калибровка лучше (`calibration_curve` ближе к диагонали); `log_loss` ниже. В комментариях поясните: чем `isotonic` отличается от `sigmoid` (нелинейный vs линейный); когда `isotonic` лучше (`n` большое); почему калибровка требует отдельного validation set (`cv="prefit"`); почему это можно делать **только** для моделей с `predict_proba`. Подсказка: `CalibratedClassifierCV(RandomForestClassifier(), method="isotonic", cv=5)`.

**Задача 100 (комбинирование: `class_weight` + calibration).** Сравните 4 варианта: обычная `LogReg`, `class_weight="balanced"`, `CalibratedClassifierCV`, `class_weight` + calibration. Ожидаемый вывод: `class_weight` улучшает recall, calibration — `log_loss`; комбинация — оптимум. В комментариях поясните: почему `class_weight` и calibration решают **разные** задачи (recall vs вероятности); как выбрать (бизнес-требования: если нужны вероятности — calibration; если только класс — `class_weight`); почему «одна метрика» не решает всё. Подсказка: сравните 4 модели через `cross_validate` с `scoring=["f1", "neg_log_loss"]`.

---

## 🔴 Уровень 14. Pipeline, production, saving (101–106)

**Что изучается:** production `Pipeline`, `ColumnTransformer` внутри, кастомный трансформер, `joblib`/`pickle`, `FunctionTransformer`, `predict.py`, `handle_unknown`, версионирование.

**Задача 101 (production Pipeline).** Соберите полный `Pipeline`: `ColumnTransformer` (imputer + scaler для числовых, one-hot для категориальных) → `HistGradientBoosting`. Обучите, оцените, сделайте `predict` на новых данных. Ожидаемый вывод: `pipeline.predict(new_data)` работает end-to-end. В комментариях поясните: почему `Pipeline` обязателен в production (нет leakage, одна строка для inference); как `ColumnTransformer` внутри `Pipeline` работает; почему `predict` на новых данных не требует ручного препроцессинга; как `pipeline.named_steps` помогает дебажить. Подсказка: `pipe = Pipeline([("prep", ColumnTransformer(...)), ("model", HistGradientBoostingClassifier())])`.

**Задача 102 (`joblib`: сохранение модели).** Сохраните обученный `Pipeline` через `joblib.dump` и загрузите через `joblib.load`. Ожидаемый вывод: `loaded_pipeline.predict(X_test)` даёт те же предсказания. В комментариях поясните: зачем сохранять модель (не переобучать каждый раз); чем `joblib` лучше `pickle` для sklearn (эффективнее для numpy-массивов); почему `pickle` небезопасен (задача 86 SQL); что сохранять (`Pipeline`, не отдельные шаги); как версионировать модели (`model_v1.joblib`). Подсказка: `import joblib; joblib.dump(pipeline, "model.joblib"); joblib.load("model.joblib")`.

**Задача 103 (кастомный трансформер).** Напишите `class DateFeatures(BaseEstimator, TransformerMixin)`, который извлекает `year, month, day_of_week` из `loan_date`. Подключите в `Pipeline`. Ожидаемый вывод: `Pipeline` работает с кастомным шагом. В комментариях поясните: зачем кастомные трансформеры (сложные преобразования, бизнес-логика); почему наследоваться от `BaseEstimator, TransformerMixin` (`get_params`, `fit_transform` бесплатно); как `fit`/`transform` должны работать (без leakage); как тестировать кастомный трансформер (`check_estimator`). Подсказка: `from sklearn.base import BaseEstimator, TransformerMixin; class DateFeatures(BaseEstimator, TransformerMixin): def fit(self, X, y=None): return self; def transform(self, X): ...`.

**Задача 104 (`FunctionTransformer`).** Примените `np.log1p` к `fine` через `FunctionTransformer`. Ожидаемый вывод: распределение `fine` становится ближе к нормальному; `R²` линейной модели растёт. В комментариях поясните: зачем `FunctionTransformer` (быстрое преобразование без класса); когда лог-преобразование полезно (правый хвост, мультипликативные эффекты); почему `log1p` (а не `log`, чтобы `log(0)` не падал); что такое `inverse_func` (обратное преобразование). Подсказка: `from sklearn.preprocessing import FunctionTransformer; FunctionTransformer(np.log1p, inverse_func=np.expm1)`.

**Задача 105 (`handle_unknown="ignore"`).** В test появляется новая категория (`genre = "комикс"`), которой не было в train. Покажите, что `OneHotEncoder(handle_unknown="ignore")` не падает, а `handle_unknown="error"` — падает. Ожидаемый вывод: `ignore` → новый столбец `0`; `error` → `ValueError`. В комментариях поясните: почему новые категории в production — норма; почему `ignore` безопаснее `error`; что происходит с признаком (все нули — модель «не знает»); что такое `min_frequency` (редкие категории → «other»). Подсказка: `OneHotEncoder(handle_unknown="ignore", min_frequency=10)`.

**Задача 106 (`predict.py`: end-to-end inference).** Сохраните `Pipeline` через `joblib`. Напишите `predict.py`, который загружает модель, читает `new_loans.csv`, делает `predict`, сохраняет `predictions.csv`. Ожидаемый вывод: `python predict.py` создаёт файл с предсказаниями. В комментариях поясните: почему inference должен быть **отдельным** скриптом (не переобучать каждый раз); что делать, если новые данные содержат невиданные категории (`handle_unknown="ignore"`); почему `Pipeline` гарантирует, что препроцессинг совпадает с train; как обрабатывать `NaN` в новых данных (imputer внутри `Pipeline`); как версионировать модель и данные. Подсказка: `pipeline = joblib.load("model.joblib"); df = pd.read_csv("new_loans.csv"); predictions = pipeline.predict(df); pd.DataFrame(predictions).to_csv("predictions.csv")`.

---

## ⚫ Уровень 15. Интерпретация (107–111)

**Что изучается:** `partial_dependence`, `SHAP`, `permutation_importance`, `feature_importances_`, `ICE`, `PDP`, объяснение предсказаний.

**Задача 107 (`partial_dependence`).** Постройте `PartialDependenceDisplay` для топ-3 признаков. Ожидаемый вывод: графики «признак → вероятность». В комментариях поясните: что такое partial dependence (средний эффект признака на предсказание); почему это **не** причинность (correlation ≠ causation); почему PDP может «врать» при коррелированных признаках (нужен `ICE`); как использовать для объяснения бизнесу. Подсказка: `from sklearn.inspection import PartialDependenceDisplay; PartialDependenceDisplay.from_estimator(model, X, features=["days_held", "reader_age"])`.

**Задача 108 (`ICE`).** Постройте ICE-plot для одного признака — увидите «индивидуальные» эффекты. Ожидаемый вывод: ICE-кривые могут сильно различаться (гетерогенный эффект); PDP — усреднение. В комментариях поясните: чем ICE отличается от PDP (индивидуальные vs средние); почему ICE полезен (находит «подгруппы» с разным эффектом); как читать (пересечения, кластеры); когда PDP «врёт», а ICE — нет. Подсказка: `PartialDependenceDisplay.from_estimator(model, X, features=["days_held"], kind="individual")`.

**Задача 109 (`SHAP`: обзорно).** Вычислите `SHAP`-values для 5 примеров через `shap` (для `RandomForest`/`HistGradientBoosting`). Ожидаемый вывод: `shap_values` для каждого признака; summary plot. В комментариях поясните: что такое SHAP (Shapley values из теории игр); почему SHAP честнее `feature_importances_` (учитывает взаимодействия); как читать summary plot (каждый признак → вклад в предсказание); почему SHAP дорогой (`O(2^n)` для точного, `TreeSHAP` — быстрее); когда SHAP обязателен (медицина, финансы — объяснимость). Подсказка: `import shap; explainer = shap.TreeExplainer(model); shap_values = explainer.shap_values(X_test)`.

**Задача 110 (`SHAP`: waterfall для одного предсказания).** Для одного примера постройте `waterfall_plot`. Ожидаемый вывод: «базовое значение + вклад каждого признака = предсказание». В комментариях поясните: что такое `base_value` (среднее предсказание); как читать waterfall (направление и величина вклада); как объяснить клиенту «почему модель так решила»; почему SHAP-объяснения — «приближение», а не «истина». Подсказка: `shap.plots.waterfall(shap_values[0])`.

**Задача 111 (сравнение интерпретаций).** Сравните `feature_importances_`, `permutation_importance`, `SHAP` на одной модели. Ожидаемый вывод: rankings часто совпадают, но есть отличия (SHAP ловит взаимодействия, PFI — «глобально», FI — biased). В комментариях поясните: когда какую использовать (`feature_importances_` — быстро, `PFI` — честно, `SHAP` — глубоко); почему rankings **не совпадают**; почему для бизнеса лучше SHAP (объяснение конкретного предсказания); почему `PFI` — «золотая середина». Подсказка: постройте три bar-plot и сравните.

---

## ⚫ Уровень 16. Продвинутые темы (112–117)

**Что изучается:** `MultiOutputClassifier`, текст (`TfidfVectorizer`, `Pipeline` для NLP), временные ряды (`TimeSeriesSplit` + `HistGradientBoosting`), `TransformedTargetRegressor`, `BaggingRegressor`, `CalibratedClassifierCV` + `Pipeline`.

**Задача 112 (`MultiOutputClassifier`).** Обучите модель, предсказывающую два таргета: `returned_on_time` и `paid_on_time`. Ожидаемый вывод: `predict` возвращает 2 столбца. В комментариях поясните: что такое multi-output (несколько таргетов); как работает `MultiOutputClassifier` (отдельная модель на каждый таргет); почему нельзя просто «объединить» таргеты; когда `ClassifierChain` лучше (`Classifiers` учитывают зависимости между таргетами). Подсказка: `from sklearn.multioutput import MultiOutputClassifier; MultiOutputClassifier(RandomForestClassifier())`.

**Задача 113 (текст: `TfidfVectorizer` + `Pipeline`).** Классифицируйте жанр по описанию книги. Ожидаемый вывод: `accuracy = 0.82` на тексте. В комментариях поясните: чем `TfidfVectorizer` лучше `CountVectorizer` (учитывает «редкость» слов); что такое `ngram_range=(1, 2)` (биграммы); зачем `stop_words="english"`/`"russian"`; почему текст → `LogisticRegression` — сильный baseline; когда нужны `transformers` (BERT, ruBERT). Подсказка: `Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)), ("model", LogisticRegression())])`.

**Задача 114 (временные ряды).** Постройте модель прогноза `fine` по датам с `TimeSeriesSplit` + `HistGradientBoostingRegressor`. Ожидаемый вывод: `RMSE` на последних 20% — лучше, чем `LinearRegression`. В комментариях поясните: почему `TimeSeriesSplit` обязателен (data leakage); что такое «лаговые признаки» (`shift`); почему boosting с лагами — хороший baseline для временных рядов; когда нужны `ARIMA`/`Prophet` (сезонность, тренд). Подсказка: `df["lag_1"] = df["fine"].shift(1); TimeSeriesSplit(n_splits=5)`.

**Задача 115 (`TransformedTargetRegressor`).** Обучите `LinearRegression` на `log(fine)` через `TransformedTargetRegressor`. Ожидаемый вывод: `R²` выше, чем на «сыром» `fine`. В комментариях поясните: зачем трансформировать `y` (правый хвост → нормальность); что делает `TransformedTargetRegressor` (fit на `log(y)`, predict на `exp`); почему `log1p` (не `log`) безопаснее; когда это вредит (интерпретация коэффициентов). Подсказка: `from sklearn.compose import TransformedTargetRegressor; TransformedTargetRegressor(regressor=LinearRegression(), func=np.log1p, inverse_func=np.expm1)`.

**Задача 116 (`BaggingRegressor`).** Обучите `BaggingRegressor(base_estimator=DecisionTreeRegressor())`. Ожидаемый вывод: RMSE ниже одного дерева, но выше `RandomForest`. В комментариях поясните: чем bagging отличается от `RandomForest` (RF — случайные признаки); когда `Bagging` полезен (любая модель как base); что такое `max_features` в bagging (доля признаков). Подсказка: `from sklearn.ensemble import BaggingRegressor`.

**Задача 117 (`CalibratedClassifierCV` + `Pipeline`).** Соберите `Pipeline` с калибровкой внутри. Ожидаемый вывод: `predict_proba` калиброваны, `log_loss` ниже. В комментариях поясните: как `CalibratedClassifierCV` работает с `Pipeline` (`cv="prefit"` — обучение на train, калибровка на отдельном); почему калибровка внутри `Pipeline` — стандарт production; что такое `brier_score_loss` (ещё одна метрика калибровки). Подсказка: `Pipeline([("prep", ...), ("cal", CalibratedClassifierCV(HistGradientBoostingClassifier(), cv=5))])`.

---

## ⚫ Уровень 17. Финальный проект (118–122)

**Что изучается:** end-to-end ML-проект — EDA → препроцессинг → baseline → модель → тюнинг → интерпретация → сохранение → отчёт.

**Задача 118 (EDA).** Проведите EDA: `df.info()`, `describe()`, `isna().sum()`, `duplicated().sum()`, распределения, корреляции. Ожидаемый вывод: 5–7 графиков + выводы. В комментариях поясните: зачем EDA (понять данные до модели); на что смотреть (пропуски, выбросы, дисбаланс, корреляции); почему «сразу в модель» — плохо; какие графики обязательны (`histplot`, `boxplot`, `heatmap`). Подсказка: `sns.heatmap(df.corr(), annot=True); sns.histplot(df["days_held"])`.

**Задача 119 (baseline + split).** Разделите данные (`train_test_split`, `stratify`, `random_state=42`). Обучите `DummyClassifier` — baseline. Ожидаемый вывод: `baseline accuracy = 0.58`. В комментариях поясните: зачем baseline (модель должна его **побить**); почему `stratify` важен; почему test нельзя трогать до финала. Подсказка: `DummyClassifier(strategy="most_frequent").fit(X_train, y_train).score(X_test, y_test)`.

**Задача 120 (сравнение 5 моделей + тюнинг).** Сравните `LogisticRegression`, `RandomForest`, `GradientBoosting`, `HistGradientBoosting`, `SVC` через `cross_val_score`. Настройте лучшую через `GridSearchCV` или `RandomizedSearchCV`. Ожидаемый вывод: таблица `model × accuracy × F1 × время`; `best_params`. В комментариях поясните: почему сравнение на CV, не на test; почему время обучения важно; как выбрать финальную модель (не только accuracy — скорость, интерпретируемость); почему `HistGradientBoosting` обычно выигрывает. Подсказка: `results = {name: cross_val_score(model, X, y, cv=5).mean() for name, model in models.items()}`.

**Задача 121 (интерпретация + финальная оценка).** Постройте `permutation_importance` / `SHAP` для финальной модели. Оцените на отложенном test: `classification_report`, `confusion_matrix`, `ROC-AUC`. Ожидаемый вывод: топ-5 признаков; `accuracy = 0.83`, `F1 = 0.80`, `AUC = 0.88`. В комментариях поясните: какие признаки важны и почему (бизнес-логика); как объяснить модель бизнесу; почему «важность» ≠ «причинность»; почему test нельзя трогать до финала. Подсказка: `permutation_importance(model, X_test, y_test); classification_report(y_test, y_pred)`.

**Задача 122 (финальный проект «под ключ»).** Соберите всё в `ml_project.py` + `notebook.ipynb` + `README.md`: (1) EDA; (2) препроцессинг в `Pipeline`; (3) baseline; (4) сравнение 5 моделей; (5) `GridSearchCV`; (6) интерпретация; (7) финальная оценка на test; (8) сохранение через `joblib`; (9) `predict.py` для inference; (10) `README` с метриками и инструкцией запуска. Ожидаемый результат: работающий ML-проект «под ключ». В комментариях опишите: как разбить на модули (`data.py`, `preprocess.py`, `train.py`, `evaluate.py`, `predict.py`); почему тесты важны (`pytest` на `Pipeline`); как воспроизвести (`seed`, `requirements.txt`); какие следующие шаги (деплой через FastAPI/Docker, мониторинг, retraining). Подсказка: начните с `notebook.ipynb` для EDA, потом переносите в `.py`-модули.

---